# Stage 3. Dual-Path Feature Extraction

Notebook ini mengekstrak fitur hand-crafted warna dan deep embedding ResNet18-CSA dari ROI palm yang sudah dinormalisasi pada stage sebelumnya, memakai modul src.common.features yang sama persis dengan situs konjungtiva karena kedua modul tersebut situs-agnostik. Backbone dipakai sebagai frozen feature extractor sesuai keputusan yang telah dikonfirmasi, tanpa stage fine-tune end-to-end terpisah.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd
import matplotlib.pyplot as plt

from configs import paths
from src.common import features

output_dir = paths.outputs_dir("palm")
manifest = pd.read_csv(output_dir / "manifest.csv")
manifest = manifest[manifest["roi_precropped"]].reset_index(drop=True)
print("sampel dengan ROI", len(manifest))

## Handcrafted Color Features

Fitur mencakup statistik RGB, rasio terkait hemoglobin, statistik HSV dan CIELAB, erythema index, tekstur gray level, dan entropy, dihitung hanya pada pixel valid hasil normalisasi Stage 2.

In [ ]:
handcrafted = features.extract_handcrafted_features(manifest)
handcrafted.head()

## Correlation with Hemoglobin

Korelasi fitur hand-crafted kunci terhadap hemoglobin memverifikasi arah hubungan sesuai fisika optik, yaitu makin rendah hemoglobin makin sedikit absorpsi merah sehingga red_ratio dan mean_a (redness CIELAB) turun.

In [ ]:
merged = manifest[["uid", "hb_gdl"]].merge(handcrafted, on="uid")
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, column in zip(axes, ["red_ratio", "mean_a", "erythema_index"]):
    ax.scatter(merged["hb_gdl"], merged[column], alpha=0.4, s=10)
    ax.set_xlabel("Hemoglobin (g/dL)")
    ax.set_ylabel(column)
    correlation = merged["hb_gdl"].corr(merged[column])
    ax.set_title(f"{column} (r={correlation:.2f})")
plt.tight_layout()
plt.show()

## Deep Embedding Extraction

Embedding ResNet18-CSA diekstrak sebagai fitur statis dengan bobot backbone pralatih ImageNet, dipakai apa adanya (frozen) sebagai bagian dari vektor fusi pada training multi-task stage berikutnya.

In [ ]:
backbone = features.EmbeddingBackbone(backbone_name="resnet18")
deep_embeddings, embedding_uids = features.extract_deep_embeddings(manifest, model=backbone)
print("embedding shape", deep_embeddings.shape)

## Fusion Attention Demonstration

Verifikasi bentuk vektor fusi (handcrafted terstandardisasi, deep embedding, demografi, dan site token) sesuai kontrak yang dipakai src.common.train.run_kfold pada stage berikutnya.

In [ ]:
fusion_input = features.build_fusion_input(handcrafted, deep_embeddings, manifest)
print("fusion input shape", fusion_input.shape)

## Save Features

In [ ]:
saved_paths = features.save_features(handcrafted, deep_embeddings, embedding_uids, output_dir=output_dir)
saved_paths